# Regression of Vote Characteristics on CAR

In [4]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')

In [5]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. 


In [6]:
%%stata

eststo clear

foreach stage in created end {
    import delimited using "$PROCESSED_DATA/event_study_panel_`stage'.csv", clear

    * Parse date
    gen day = date(date, "YMD")
    format day %td
    gen year = year(day)

    * Ensure index is numeric (safeguard if it came in as string)
    capture confirm numeric variable index
    if _rc {
        destring index, replace ignore("[]")
    }

    * Encode token for FE/clustering
    encode gecko_id, gen(token)

    * Label variable
    label var non_whale_victory_vn "\${\it Small Win}_{i,t}\$"

    * Keep CAR[-5,5] window
    keep if index == 5

    * Run regression
    reghdfe car non_whale_victory_vn, absorb(year token) vce(cluster token)
    eststo car_`stage'
    estadd local fe_token "Y"
    estadd local fe_time "Y"
}

* Export combined LaTeX table
esttab                                                           ///
    car_created car_end                                          ///
    using "$TABLES/reg_car_win.tex", replace                     ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs                                             ///
    b(%9.3f) se(%9.2f)                                           ///
    mtitles("\${\it CAR^{\it Create}}_i\$" "\${\it CAR^{\it End}}_i\$") ///
    posthead(\cmidrule(lr){2-3} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_token fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
         labels("Token FE" "Year FE" "Observations" "Adjusted R²"))


. 
. eststo clear

. 
. foreach stage in created end {
  2.     import delimited using "$PROCESSED_DATA/event_study_panel_`stage'.csv"
> , clear
  3. 
.     * Parse date
.     gen day = date(date, "YMD")
  4.     format day %td
  5.     gen year = year(day)
  6. 
.     * Ensure index is numeric (safeguard if it came in as string)
.     capture confirm numeric variable index
  7.     if _rc {
  8.         destring index, replace ignore("[]")
  9.     }
 10. 
.     * Encode token for FE/clustering
.     encode gecko_id, gen(token)
 11. 
.     * Label variable
.     label var non_whale_victory_vn "\${\it Small Win}_{i,t}\$"
 12. 
.     * Keep CAR[-5,5] window
.     keep if index == 5
 13. 
.     * Run regression
.     reghdfe car non_whale_victory_vn, absorb(year token) vce(cluster token)
 14.     eststo car_`stage'
 15.     estadd local fe_token "Y"
 16.     estadd local fe_time "Y"
 17. }
(encoding automatically selected: ISO-8859-1)
(40 vars, 17,523 obs)
(15,930 observations deleted)
